# OpenPlaque — Left-Coronary Through-Vessel Continuity v1
Fresh-baseline topology adjudication. Research use only; frozen master is never modified.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, sys, shutil, importlib
DRIVE_ROOT=Path('/content/drive/MyDrive/OpenPlaque')
OUTPUT=DRIVE_ROOT/'Left_Coronary_Through_Vessel_Continuity_v1'
OUTPUT.mkdir(parents=True,exist_ok=True)
BRANCH='left-coronary-through-vessel-continuity-from-main'
PIN='1521f331a3f96c46cfa7ccbfe383265641bff55a'
BASELINE='0593b453959f5a353d644267fbeef24b514ef4d7'
REUSE_VALID_CACHE=True
(OUTPUT/'notebook_started.json').write_text(json.dumps({'branch':BRANCH,'pin':PIN,'baseline':BASELINE,'reuse_valid_cache':REUSE_VALID_CACHE},indent=2))
print('Output:',OUTPUT)
print('Science pin:',PIN)

In [ ]:
REPO=Path('/content/OpenPlaque')
if REPO.exists(): shutil.rmtree(REPO)
assert os.system(f'git clone -q --branch {BRANCH} https://github.com/pazzani/OpenPlaque.git {REPO}')==0
assert os.system(f'git -C {REPO} checkout -q {PIN}')==0
HEAD=os.popen(f'git -C {REPO} rev-parse HEAD').read().strip()
MERGE_BASE=os.popen(f'git -C {REPO} merge-base HEAD {BASELINE}').read().strip()
print('HEAD',HEAD); print('merge-base',MERGE_BASE)
assert HEAD==PIN
assert MERGE_BASE==BASELINE
assert os.system(f'{sys.executable} -m pip install -q {REPO}')==0
for name in list(sys.modules):
    if name=='openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
importlib.invalidate_caches()

In [ ]:
import py_compile, pytest
src=REPO/'src/openplaque/left_coronary_through_vessel_continuity_v1.py'
py_compile.compile(str(src),doraise=True)
from openplaque.left_coronary_through_vessel_continuity_v1 import synthetic_continuity_self_test
print('Synthetic:',synthetic_continuity_self_test())
rc=pytest.main(['-q',str(REPO/'tests/test_left_coronary_through_vessel_continuity_v1.py')])
assert rc==0, f'pytest failed: {rc}'

In [ ]:
required=[
 DRIVE_ROOT/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
 DRIVE_ROOT/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
 DRIVE_ROOT/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 DRIVE_ROOT/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
 DRIVE_ROOT/'Left_Proximal_Trunk_Continuation_QC_v1/accepted_proximal_trunk_continuation_candidate.csv',
 DRIVE_ROOT/'Joint_Three_Vessel_Template_Classifier_v1/candidate_04_source_path.csv',
 DRIVE_ROOT/'LCX_Distal_Reacquisition_v1_fixed/C7_extended_path.csv',
 DRIVE_ROOT/'Left_Coronary_Bifurcation_Parent_Recovery_v1/summary.json',
 DRIVE_ROOT/'Left_Coronary_Bifurcation_Parent_Recovery_v1/best_target_parent_path.csv'
]
missing=[str(p) for p in required if not p.exists()]
assert not missing, 'Missing prerequisites:\n'+'\n'.join(missing)
(OUTPUT/'preflight_complete.json').write_text(json.dumps({'ok':True,'n_required':len(required),'pin':PIN},indent=2))
print('Preflight complete')

In [ ]:
from openplaque.left_coronary_through_vessel_continuity_v1 import run
result=run(str(DRIVE_ROOT),str(OUTPUT))
s=result['summary']
print('STATUS:',s['status'])
print('Through-vessel deflection deg:',s.get('junction',{}).get('through_deflection_deg'))
print('Dense QC pass fraction:',s.get('dense_qc',{}).get('pass_fraction'))
print('C6/C7 split control:',s.get('C67_split_control',{}).get('control_gate_pass'))
print('Prior parent re-enters LAD:',s.get('parent_candidate_reentry',{}).get('reentry_gate_pass'))
print('Parent overlap within 2 mm:',s.get('parent_candidate_reentry',{}).get('fraction_within_2mm'))
print('Parent contiguous span within 2 mm:',s.get('parent_candidate_reentry',{}).get('max_contiguous_span_within_2mm'))
print('Report:',result['report'])
print('ZIP:',result['zip'])